In [14]:
# imporitng libraries
import os
import joblib
import torch
import pyiqa
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [15]:
# configuring paths
ROOT_IMAGES = "dataset_split"
ROOT_EMBEDDINGS = "embeddings"

In [16]:
# loading our traineed svm model
svm = joblib.load("svm_rejection_model.pkl")
scaler = joblib.load("svm_scaler.pkl")
print("SVM Loaded")

SVM Loaded


In [17]:
# loading MUSIQ for quality check of test images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
musiq_metric = pyiqa.create_metric("musiq", device=device)
print("MUSIQ Loaded")

cpu
Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth
MUSIQ Loaded


In [18]:
# quality score function
def get_quality_score(image_path):
    score = musiq_metric(image_path)
    return float(score.cpu().item())

In [19]:
# loading gallery database
def load_gallery_db(gallery_root):
    gallery_db = {}
    total = 0

    for identity in os.listdir(gallery_root):
        identity_dir = os.path.join(gallery_root, identity)
        gallery_db[identity] = []

        for file in os.listdir(identity_dir):
            emb = np.load(os.path.join(identity_dir, file))
            gallery_db[identity].append(emb)
            total += 1

    print("Gallery Embeddings Loaded:", total)
    return gallery_db

In [20]:
# gallery search
def search_gallery(probe_embedding, gallery_db):
    identity_scores = {}

    for identity in gallery_db:
        sims = []

        for gallery_emb in gallery_db[identity]:
            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]
            sims.append(sim)
        identity_scores[identity] = max(sims)

    sorted_scores = sorted(identity_scores.items(), key=lambda x: x[1], reverse=True)
    predicted_identity = sorted_scores[0][0]
    best_similarity = sorted_scores[0][1]
    second_similarity = sorted_scores[1][1]
    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)

In [21]:
# loading test gallery
gallery_db = load_gallery_db(os.path.join(ROOT_EMBEDDINGS, "test", "gallery"))

Gallery Embeddings Loaded: 482


In [22]:
# testing
probe_emb_root = os.path.join(ROOT_EMBEDDINGS, "test", "degraded_probes")
probe_img_root = os.path.join(ROOT_IMAGES, "test", "degraded_probes")

results = []

counter = 0

for identity in tqdm(os.listdir(probe_emb_root)):

    emb_dir = os.path.join(probe_emb_root, identity)

    img_dir = os.path.join(probe_img_root, identity)

    for emb_file in os.listdir(emb_dir):

        emb_path = os.path.join(emb_dir, emb_file)

        probe_embedding = np.load(emb_path)

        base_name = os.path.splitext(emb_file)[0]

        image_path = None

        for ext in [".jpg", ".jpeg", ".png"]:

            candidate = os.path.join(img_dir, base_name + ext)

            if os.path.exists(candidate):

                image_path = candidate

                break

        if image_path is None:

            continue

        quality_score = get_quality_score(image_path)

        (
            predicted_identity,
            best_similarity,
            second_similarity,
            margin,
        ) = search_gallery(
            probe_embedding,
            gallery_db,
        )

        features = pd.DataFrame(
            [[quality_score, best_similarity, margin]],
            columns=[
                "quality_score",
                "best_similarity",
                "margin",
            ],
        )

        features_scaled = scaler.transform(features)

        svm_decision = svm.predict(features_scaled)[0]

        confidence_score = svm.predict_proba(
            features_scaled
        )[0][1]

        if svm_decision == 1:

            final_decision = predicted_identity

        else:

            final_decision = "REJECT"

        # ground truth
        label = int(predicted_identity == identity)

        # FALSE REJECTS
        if label == 1 and svm_decision == 0:

            print("\n====================")
            print("FALSE REJECT FOUND")
            print("====================")

            print("True Identity:", identity)

            print("Closest Match:", predicted_identity)

            print("Confidence:", round(confidence_score, 4))

            print("Quality:", round(quality_score, 4))

            print("Best Similarity:", round(best_similarity, 4))

            print("Second Similarity:", round(second_similarity, 4))

            print("Margin:", round(margin, 4))

        # FALSE ACCEPTS
        if label == 0 and svm_decision == 1:

            print("\n====================")
            print("FALSE ACCEPT FOUND")
            print("====================")

            print("True Identity:", identity)

            print("Closest Match:", predicted_identity)

            print("Confidence:", round(confidence_score, 4))

            print("Quality:", round(quality_score, 4))

            print("Best Similarity:", round(best_similarity, 4))

            print("Second Similarity:", round(second_similarity, 4))

            print("Margin:", round(margin, 4))

        results.append(
            {
                "probe_image": base_name,
                "true_identity": identity,
                "predicted_identity": predicted_identity,
                "final_decision": final_decision,
                "quality_score": quality_score,
                "best_similarity": best_similarity,
                "second_best_similarity": second_similarity,
                "margin": margin,
                "confidence_score": confidence_score,
                "svm_decision": svm_decision,
                "label": label,
            }
        )

        counter += 1

        if counter % 50 == 0:

            print(f"\nProcessed {counter}")

            print("Truth:", identity)

            print("Closest Match:", predicted_identity)

            print("Final Decision:", final_decision)

            print("Confidence:", round(confidence_score, 4))

            print("Quality:", round(quality_score, 4))

            print("Similarity:", round(best_similarity, 4))

            print(
                "Second Best Similarity:",
                round(second_similarity, 4),
            )

            print("Margin:", round(margin, 4))

            print(
                "Accept"
                if svm_decision == 1
                else "Reject"
            )

 10%|█         | 3/30 [00:08<01:14,  2.77s/it]


Processed 50
Truth: Angelina_Jolie
Closest Match: Angelina_Jolie
Final Decision: Angelina_Jolie
Confidence: 0.994
Quality: 30.7015
Similarity: 0.6188
Second Best Similarity: 0.1593
Margin: 0.4595
Accept


 20%|██        | 6/30 [00:16<00:57,  2.38s/it]


FALSE REJECT FOUND
True Identity: Anna_Kournikova
Closest Match: Anna_Kournikova
Confidence: 0.0804
Quality: 24.269
Best Similarity: 0.3475
Second Similarity: 0.2337
Margin: 0.1138

FALSE REJECT FOUND
True Identity: Anna_Kournikova
Closest Match: Anna_Kournikova
Confidence: 0.0022
Quality: 14.4702
Best Similarity: 0.1932
Second Similarity: 0.1817
Margin: 0.0114

FALSE REJECT FOUND
True Identity: Anna_Kournikova
Closest Match: Anna_Kournikova
Confidence: 0.0119
Quality: 18.4484
Best Similarity: 0.2968
Second Similarity: 0.2236
Margin: 0.0731

FALSE REJECT FOUND
True Identity: Anna_Kournikova
Closest Match: Anna_Kournikova
Confidence: 0.0241
Quality: 17.2319
Best Similarity: 0.3044
Second Similarity: 0.2008
Margin: 0.1037

FALSE REJECT FOUND
True Identity: Anna_Kournikova
Closest Match: Anna_Kournikova
Confidence: 0.6482
Quality: 14.0798
Best Similarity: 0.3798
Second Similarity: 0.191
Margin: 0.1888


 23%|██▎       | 7/30 [00:18<00:53,  2.31s/it]

...

 90%|█████████ | 27/30 [01:29<00:10,  3.41s/it]


Processed 500
Truth: Megawati_Sukarnoputri
Closest Match: Megawati_Sukarnoputri
Final Decision: Megawati_Sukarnoputri
Confidence: 0.9748
Quality: 20.0234
Similarity: 0.7018
Second Best Similarity: 0.2105
Margin: 0.4913
Accept


 97%|█████████▋| 29/30 [01:37<00:03,  3.52s/it]


FALSE REJECT FOUND
True Identity: Pervez_Musharraf
Closest Match: Pervez_Musharraf
Confidence: 0.1519
Quality: 13.1415
Best Similarity: 0.3333
Second Similarity: 0.1786
Margin: 0.1547


100%|██████████| 30/30 [01:40<00:00,  3.35s/it]


In [23]:
# saving predictions
test_df = pd.DataFrame(results)
test_df.to_csv("test_predictions.csv", index=False)
print(test_df.shape)
print("Saved test_predictions.csv")

(540, 11)
Saved test_predictions.csv


In [24]:
# overall accuracy
identity_accuracy = test_df["label"].mean()
print("Identity Accuracy:", identity_accuracy)

Identity Accuracy: 0.9777777777777777


In [25]:
# svm performance
y_true = test_df["label"]
y_pred = test_df["svm_decision"]

print("Accept/Reject Accuracy:", accuracy_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accept/Reject Accuracy: 0.9740740740740741
[[ 11   1]
 [ 13 515]]
              precision    recall  f1-score   support

           0       0.46      0.92      0.61        12
           1       1.00      0.98      0.99       528

    accuracy                           0.97       540
   macro avg       0.73      0.95      0.80       540
weighted avg       0.99      0.97      0.98       540



In [26]:
# FAR / FRR / TAR / TRR

cm = confusion_matrix(y_true, y_pred)

TN, FP, FN, TP = cm.ravel()

print("\n===== CONFUSION MATRIX VALUES =====")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

# True Acceptance Rate
TAR = TP / (TP + FN)

# False Rejection Rate
FRR = FN / (TP + FN)

# True Rejection Rate
TRR = TN / (TN + FP)

# False Acceptance Rate
FAR = FP / (TN + FP)

print("\n===== VERIFICATION METRICS =====")

print("TAR (True Acceptance Rate):", TAR)
print("FRR (False Rejection Rate):", FRR)

print("TRR (True Rejection Rate):", TRR)
print("FAR (False Acceptance Rate):", FAR)


===== CONFUSION MATRIX VALUES =====
TP: 515
TN: 11
FP: 1
FN: 13

===== VERIFICATION METRICS =====
TAR (True Acceptance Rate): 0.9753787878787878
FRR (False Rejection Rate): 0.02462121212121212
TRR (True Rejection Rate): 0.9166666666666666
FAR (False Acceptance Rate): 0.08333333333333333
